In [1]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

In [2]:
spark = SparkSession.builder.master("local[*]").getOrCreate()

In [3]:
df = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv("data/pokeData.csv")
)

In [7]:
df.show()

+----------+----------+--------+-------------------+----------+
|pokemon_id|      name|height_m|         updated_at|batch_date|
+----------+----------+--------+-------------------+----------+
|         1| Bulbasaur|     0.7|2026-10-10 09:00:00|2026-10-10|
|         2|   Ivysaur|     1.0|2026-10-10 09:15:00|2026-10-10|
|         3|  Venusaur|     2.0|2026-10-10 09:30:00|2026-10-10|
|         4|Charmander|     0.6|2026-10-10 10:00:00|2026-10-10|
|         5|Charmeleon|     1.1|2026-10-10 10:15:00|2026-10-10|
|         6| Charizard|     1.7|2026-10-10 10:30:00|2026-10-10|
|         7|  Squirtle|     0.5|2026-10-11 09:00:00|2026-10-11|
|         8| Wartortle|     1.0|2026-10-11 09:15:00|2026-10-11|
|         9| Blastoise|     1.6|2026-10-11 09:30:00|2026-10-11|
|        10|  Caterpie|     0.3|2026-10-11 10:00:00|2026-10-11|
|         3|  Venusaur|     2.0|2026-10-12 08:00:00|2026-10-12|
|         4|Charmander|     0.7|2026-10-12 08:30:00|2026-10-12|
|         5|Charmeleon|     1.2|2026-10-

In [4]:
window = Window.partitionBy("pokemon_id").orderBy(
    F.col("updated_at").desc()
)

In [5]:
result = (
    df.withColumn("row_num", F.row_number().over(window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

In [6]:
result.orderBy("pokemon_id").show(truncate=False)

+----------+----------+--------+-------------------+----------+
|pokemon_id|name      |height_m|updated_at         |batch_date|
+----------+----------+--------+-------------------+----------+
|1         |Bulbasaur |0.7     |2026-10-10 09:00:00|2026-10-10|
|2         |Ivysaur   |1.0     |2026-10-10 09:15:00|2026-10-10|
|3         |Venusaur  |2.0     |2026-10-12 08:00:00|2026-10-12|
|4         |Charmander|0.8     |2026-10-14 08:00:00|2026-10-14|
|5         |Charmeleon|1.2     |2026-10-12 09:00:00|2026-10-12|
|6         |Charizard |1.7     |2026-10-10 10:30:00|2026-10-10|
|7         |Squirtle  |0.5     |2026-10-11 09:00:00|2026-10-11|
|8         |Wartortle |1.1     |2026-10-14 08:30:00|2026-10-14|
|9         |Blastoise |1.6     |2026-10-11 09:30:00|2026-10-11|
|10        |Caterpie  |0.3     |2026-10-11 10:00:00|2026-10-11|
|11        |Metapod   |0.7     |2026-10-12 09:30:00|2026-10-12|
|12        |Butterfree|1.1     |2026-10-12 10:00:00|2026-10-12|
|13        |Weedle    |0.3     |2026-10-

In [8]:
spark.stop()